# Multi-agent ant-maze for teacher-studnet training

In [7]:
import os

# MuJoCo (used by brax.io.image) must know the GL backend before `import mujoco`.
# On headless Linux or SSH without DISPLAY, use egl (needs NVIDIA drivers + libEGL).
# On a local desktop with a display, you can use: os.environ["MUJOCO_GL"] = "glfw"
if "MUJOCO_GL" not in os.environ:
    os.environ["MUJOCO_GL"] = "egl"

import copy
import xml.etree.ElementTree as ET

import jax
import mujoco
from brax import base, math
from brax.envs.base import PipelineEnv, State
from brax.io import mjcf
from jax import numpy as jnp
import numpy as np
import random
import matplotlib.pyplot as plt
from brax.io import html
from brax.io import image
from multi_ant_mjcf import build_multi_ant_maze_root, world_agent_conaffinity_mask

In [8]:
RESET = R = "r"
GOAL = G = "g"


U_MAZE = [
    [1, 1, 1, 1, 1],
    [1, R, G, G, 1],
    [1, 1, 1, G, 1],
    [1, G, G, G, 1],
    [1, 1, 1, 1, 1],
]

U_MAZE_1 = [
    [1, 1, 1, 1, 1],
    [1, R, G, 0, 1],
    [1, 1, 1, 0, 1],
    [1, 0, 0, 0, 1],
    [1, 1, 1, 1, 1],
]
U_MAZE_2 = [
    [1, 1, 1, 1, 1],
    [1, R, G, G, 1],
    [1, 1, 1, 0, 1],
    [1, 0, 0, 0, 1],
    [1, 1, 1, 1, 1],
]
U_MAZE_3 = [
    [1, 1, 1, 1, 1],
    [1, R, G, G, 1],
    [1, 1, 1, G, 1],
    [1, 0, 0, 0, 1],
    [1, 1, 1, 1, 1],
]
U_MAZE_4 = [
    [1, 1, 1, 1, 1],
    [1, R, G, G, 1],
    [1, 1, 1, G, 1],
    [1, 0, 0, G, 1],
    [1, 1, 1, 1, 1],
]


U_MAZE_SKILL_EVAL = [
    [1, 1, 1, 1, 1],
    [1, R, G, G, 1],
    [1, 1, 1, G, 1],
    [1, G, G, G, 1],
    [1, 1, 1, 1, 1],
]
# U_MAZE = [
#     [1, 1, 1, 1, 1],
#     [1, R, R, R, 1],
#     [1, 1, 1, R, 1],
#     [1, G, R, R, 1],
#     [1, 1, 1, 1, 1],
# ]

U_MAZE_NO_GOAL = [
    [1, 1, 1, 1, 1],
    [1, R, 0, 0, 1],
    [1, 1, 1, 0, 1],
    [1, 0, 0, 0, 1],
    [1, 1, 1, 1, 1],
]


U_MAZE_50_PERCENT = [
    [1, 1, 1, 1, 1],
    [1, R, G, G, 1],
    [1, 1, 1, 0, 1],
    [1, 0, 0, 0, 1],
    [1, 1, 1, 1, 1],
]

U_MAZE_SINGLE_GOAL = [
    [1, 1, 1, 1, 1],
    [1, R, 0, 0, 1],
    [1, 1, 1, 0, 1],
    [1, G, 0, 0, 1],
    [1, 1, 1, 1, 1],
]

U_MAZE_EVAL = [
    [1, 1, 1, 1, 1],
    [1, R, 0, 0, 1],
    [1, 1, 1, 0, 1],
    [1, G, G, G, 1],
    [1, 1, 1, 1, 1],
]


BIG_MAZE = [
    [1, 1, 1, 1, 1, 1, 1, 1],
    [1, R, G, 1, 1, G, G, 1],
    [1, G, G, 1, G, G, G, 1],
    [1, 1, G, G, G, 1, 1, 1],
    [1, G, G, 1, G, G, G, 1],
    [1, G, 1, G, G, 1, G, 1],
    [1, G, G, G, 1, G, G, 1],
    [1, 1, 1, 1, 1, 1, 1, 1],
]

BIG_MAZE_SINGLE_GOAL = [
    [1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 0, 1, 1, G, G, 1],
    [1, 0, 0, 1, 0, 0, G, 1],
    [1, 1, 0, 0, 0, 1, 1, 1],
    [1, 0, 0, 1, 0, 0, 0, 1],
    [1, 0, 1, 0, 0, 1, 0, 1],
    [1, R, 0, 0, 1, 0, 0, 1],
    [1, 1, 1, 1, 1, 1, 1, 1],
]

BIG_MAZE_SINGLE_GOAL_MANY_STARTS = [
    [1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 0, 1, 1, G, G, 1],
    [1, 0, 0, 1, 0, 0, G, 1],
    [1, 1, 0, 0, 0, 1, 1, 1],
    [1, 0, 0, 1, 0, 0, 0, 1],
    [1, 0, 1, 0, 0, 1, 0, 1],
    [1, R, 0, 0, 1, 0, 0, 1],
    [1, 1, 1, 1, 1, 1, 1, 1],
]

BIG_MAZE_HARD_GOALS = [
    [1, 1, 1, 1, 1, 1, 1, 1],
    [1, R, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, G, 1, G, 1],
    [1, 1, 1, 1, G, 1, G, 1],
    [1, 1, 1, 1, 1, 1, G, 1],
    [1, 1, 1, 1, 1, 1, 1, 1],
]

BIG_MAZE_EVAL = [
    [1, 1, 1, 1, 1, 1, 1, 1],
    [1, R, 0, 1, 1, G, G, 1],
    [1, 0, 0, 1, 0, G, G, 1],
    [1, 1, 0, 0, 0, 1, 1, 1],
    [1, 0, 0, 1, 0, 0, 0, 1],
    [1, 0, 1, G, 0, 1, G, 1],
    [1, 0, G, G, 1, G, G, 1],
    [1, 1, 1, 1, 1, 1, 1, 1],
]

HARDEST_MAZE = [
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, R, G, G, G, 1, G, G, G, G, G, 1],
    [1, G, 1, 1, G, 1, G, 1, G, 1, G, 1],
    [1, G, G, G, G, G, G, 1, G, G, G, 1],
    [1, G, 1, 1, 1, 1, G, 1, 1, 1, G, 1],
    [1, G, G, 1, G, 1, G, G, G, G, G, 1],
    [1, 1, G, 1, G, 1, G, 1, G, 1, 1, 1],
    [1, G, G, 1, G, G, G, 1, G, G, G, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
]

HARDEST_MAZE_50_PERCENT = [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
                        [1, R, G, G, G, 1, 0, 0, 0, 0, 0, 1],
                        [1, G, 1, 1, G, 1, 0, 1, 0, 1, 0, 1],
                        [1, G, G, G, G, G, G, 1, 0, 0, 0, 1],
                        [1, G, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1],
                        [1, G, G, 1, 0, 1, 0, 0, 0, 0, 0, 1],
                        [1, 1, G, 1, 0, 1, 0, 1, 0, 1, 1, 1],
                        [1, G, G, 1, 0, 0, 0, 1, 0, 0, 0, 1],
                        [1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1]]


HARDEST_MAZE_SINGLE_GOAL = \
                [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
                [1, R, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1],
                [1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1],
                [1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
                [1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1],
                [1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1],
                [1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1],
                [1, 0, 0, 1, 0, 0, 0, 1, G, G, G, 1], # goal coordinate is (28, 40)
                [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]

HARDEST_MAZE_HARD_GOALS = [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
                [1, R, 0, 0, 0, 1, 0, 0, 0, 0, G, 1],
                [1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1],
                [1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
                [1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1],
                [1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1],
                [1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1],
                [1, G, 0, 1, 0, 0, 0, 1, 0, 0, G, 1], # goal coordinate is (28, 40)
                [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]



MAZE_HEIGHT = 0.5


def sample_goals(structure, rng, size_scaling=4):
    '''Sample goals given a maze structure'''
    possible_goals = find_goals(structure, size_scaling)
    idx = jax.random.randint(rng, (1,), 0, len(possible_goals))
    return jnp.array(possible_goals[idx])[0]
    


def find_starts(structure, size_scaling):
    starts = []
    for i in range(len(structure)):
        for j in range(len(structure[0])):
            if structure[i][j] == RESET:
                starts.append([i * size_scaling, j * size_scaling])

    return jnp.array(starts)


def find_goals(structure, size_scaling):
    goals = []
    for i in range(len(structure)):
        for j in range(len(structure[0])):
            if structure[i][j] == GOAL:
                goals.append([i * size_scaling, j * size_scaling])
    return jnp.array(goals)

def find_walls(structure, size_scaling):
    walls = []
    for i in range(len(structure)):
        for j in range(len(structure[0])):
            if structure[i][j] == 1:
                walls.append([i * size_scaling, j * size_scaling])
    return jnp.array(walls)

def find_floor(structure, size_scaling):
    floor = []
    for i in range(len(structure)):
        for j in range(len(structure[0])):
            if structure[i][j] == 0:
                floor.append([i * size_scaling, j * size_scaling])
    return jnp.array(floor)


# Create a xml with maze and a list of possible goal positions
def make_maze(maze_layout_name, maze_size_scaling, n_agents: int = 2):
    if maze_layout_name == "u_maze":
        maze_layout = U_MAZE
    elif maze_layout_name == "u_maze_1":
        maze_layout = U_MAZE_1
    elif maze_layout_name == "u_maze_2":
        maze_layout = U_MAZE_2
    elif maze_layout_name == "u_maze_3":
        maze_layout = U_MAZE_3
    elif maze_layout_name == "u_maze_4":
        maze_layout = U_MAZE_4
    elif maze_layout_name == "u_maze_skill_eval":
        maze_layout = U_MAZE_SKILL_EVAL
    elif maze_layout_name == "u_maze_single_goal":
        maze_layout = U_MAZE_SINGLE_GOAL
    elif maze_layout_name == "u_maze_eval":
        maze_layout = U_MAZE_EVAL
    elif maze_layout_name == "big_maze":
        maze_layout = BIG_MAZE
    elif maze_layout_name == "big_maze_single_goal":
        maze_layout = BIG_MAZE_SINGLE_GOAL
    elif maze_layout_name == "big_maze_eval":
        maze_layout = BIG_MAZE_EVAL
    elif maze_layout_name == "hardest_maze":
        maze_layout = HARDEST_MAZE
    elif maze_layout_name == "hardest_maze_50_percent":
        maze_layout = HARDEST_MAZE_50_PERCENT
    elif maze_layout_name == "hardest_maze_single_goal":
        maze_layout = HARDEST_MAZE_SINGLE_GOAL
    elif maze_layout_name == "hardest_maze_hard_goals":
        maze_layout = HARDEST_MAZE_HARD_GOALS
    elif maze_layout_name == "big_maze_single_goal_many_starts":
        maze_layout = BIG_MAZE_SINGLE_GOAL_MANY_STARTS
    else:
        raise ValueError(f"Unknown maze layout: {maze_layout_name}")

    possible_starts = find_starts(maze_layout, maze_size_scaling)
    possible_goals = find_goals(maze_layout, maze_size_scaling)
    walls = find_walls(maze_layout, maze_size_scaling)
    floor = find_floor(maze_layout, maze_size_scaling)

    base_root = build_multi_ant_maze_root(n_agents)
    wall_agent_conaff = world_agent_conaffinity_mask(n_agents)
    tree = ET.ElementTree(base_root)
    worldbody = tree.find(".//worldbody")

    for i in range(len(maze_layout)):
        for j in range(len(maze_layout[0])):
            struct = maze_layout[i][j]
            if struct == 1:
                ET.SubElement(
                    worldbody,
                    "geom",
                    name="block_%d_%d" % (i, j),
                    pos="%f %f %f"
                    % (
                        i * maze_size_scaling,
                        j * maze_size_scaling,
                        MAZE_HEIGHT / 2 * maze_size_scaling,
                    ),
                    size="%f %f %f"
                    % (
                        0.5 * maze_size_scaling,
                        0.5 * maze_size_scaling,
                        MAZE_HEIGHT / 2 * maze_size_scaling,
                    ),
                    type="box",
                    material="",
                    contype="1",
                    conaffinity=str(wall_agent_conaff),
                    rgba="0.7 0.5 0.3 1.0",
                )

    # Fixed top-down camera on the maze center (see all agents at once).
    rows, cols = len(maze_layout), len(maze_layout[0])
    cx = 0.5 * (rows - 1) * maze_size_scaling
    cy = 0.5 * (cols - 1) * maze_size_scaling
    span_xy = max(rows, cols) * maze_size_scaling
    cam_z = max(22.0, 0.95 * span_xy)
    fovy = min(85.0, 42.0 + 0.5 * span_xy)
    for _cam in list(worldbody.findall("camera")):
        if _cam.get("name") == "maze_top":
            worldbody.remove(_cam)
    ET.SubElement(
        worldbody,
        "camera",
        {
            "name": "maze_top",
            "mode": "fixed",
            "pos": f"{cx} {cy} {cam_z}",
            "xyaxes": "1 0 0 0 1 0",
            "fovy": str(fovy),
        },
    )

    tree = tree.getroot()
    xml_string = ET.tostring(tree)

    return xml_string, possible_starts, possible_goals

def show_env_img(env):
    env_img = env.render()
    plt.imshow(env_img)
    plt.show()


In [9]:
def collect_rollout(act_key, env):
    """
    Renders a given environment over a series of steps and stores the resulting
    HTML file to a specified directory. Logs the rendered HTML using wandb.

    When the environment has skill_mazes enabled, renders a separate rollout
    for each skill so the per-skill behaviour can be inspected.
    """
    exp_dir = "./multi_ant_rendering_videos"
    exp_name = "multi_ant_maze_prototyping"
    num_steps = 200
    jit_env_reset = jax.jit(env.reset)
    jit_env_step = jax.jit(env.step)
    rollout = []
    key = jax.random.PRNGKey(seed=1)
    key, subkey = jax.random.split(key)
    state = jit_env_reset(rng=subkey)
    for i in range(200):
        rollout.append(state.pipeline_state)
        key, subkey = jax.random.split(key)
        action = jax.random.uniform(key, (env.action_size, ))
        action = action[0]
        state = jit_env_step(state, action)
        if i % 1000 == 0:
            key, subkey = jax.random.split(key)
            state = jit_env_reset(rng=subkey)
    return env, rollout

In [20]:
class AntMaze(PipelineEnv):
    def __init__(
        self,
        ctrl_cost_weight=0.5,
        use_contact_forces=False,
        contact_cost_weight=5e-4,
        healthy_reward=1.0,
        terminate_when_unhealthy=True,
        healthy_z_range=(0.2, 1.0),
        contact_force_range=(-1.0, 1.0),
        reset_noise_scale=0.1,
        exclude_current_positions_from_observation=False,
        backend="generalized",
        maze_layout_name="u_maze",
        maze_size_scaling=4.0,
        dense_reward: bool = False,
        skill_mazes: bool = False,
        skill_mazes_eval: bool = True,
        n_agents: int = 2,
        agent_start_offsets=None,
        **kwargs,
    ):
        # ``n_agents`` controls MJCF generation (``make_maze`` builds N ants + one target).
        if n_agents < 1:
            raise ValueError("n_agents must be >= 1")
        self._n_agents = n_agents
        self._qpos_per_agent = 15
        self._qvel_per_agent = 14
        self._act_per_agent = 8
        if agent_start_offsets is None:
            agent_start_offsets = tuple((float(i * 1.5), 0.0) for i in range(n_agents))
        assert len(agent_start_offsets) == n_agents, (
            f"agent_start_offsets must have length n_agents={n_agents}"
        )
        self._agent_start_offsets = jnp.array(agent_start_offsets, dtype=jnp.float32)

        self.skill_mazes = skill_mazes
        self.skill_mazes_eval = skill_mazes_eval
        if self.skill_mazes:
            xml_string, possible_starts, _ = make_maze(
                maze_layout_name, maze_size_scaling, n_agents=n_agents
            )
            _, _, self.possible_goals_1 = make_maze("u_maze_1", maze_size_scaling, n_agents=n_agents)
            _, _, self.possible_goals_2 = make_maze("u_maze_2", maze_size_scaling, n_agents=n_agents)
            _, _, self.possible_goals_3 = make_maze("u_maze_3", maze_size_scaling, n_agents=n_agents)
            _, _, self.possible_goals_4 = make_maze("u_maze_4", maze_size_scaling, n_agents=n_agents)
            self.goals_skills = [self.possible_goals_1, self.possible_goals_2, self.possible_goals_3, self.possible_goals_4]
        elif self.skill_mazes_eval:
            xml_string, possible_starts, possible_goals = make_maze(
                maze_layout_name, maze_size_scaling, n_agents=n_agents
            )
            self.goals_skills = [possible_goals]
        else:
            xml_string, possible_starts, possible_goals = make_maze(
                maze_layout_name, maze_size_scaling, n_agents=n_agents
            )
            self.goals_skills = [possible_goals]
        # import pdb;pdb.set_trace()
        # print(possible_goals)
        self.maze_layout_name = maze_layout_name
        sys = mjcf.loads(xml_string)
        self.possible_starts = possible_starts
        self.possible_goals = self.possible_goals_1 if self.skill_mazes else possible_goals
        if self.skill_mazes:
            pass
        else:
            self.maze_layout_name = maze_layout_name
            if "single_goal" in self.maze_layout_name:
                self.hard_goal = jnp.array([12, 4])
            if "u_maze" in self.maze_layout_name:
                self.max_x = 15
                self.min_x = 2.5
            if "big_maze" in self.maze_layout_name:
                self.max_x = 24
                self.min_x = 2.5
            if "hardest_maze" in self.maze_layout_name:
                self.max_x = 45
                self.min_x = 2.5


        n_frames = 5

        if backend in ["spring", "positional"]:
            sys = sys.tree_replace({"opt.timestep": 0.005})
            n_frames = 10

        if backend == "mjx":
            sys = sys.tree_replace(
                {
                    "opt.solver": mujoco.mjtSolver.mjSOL_NEWTON,
                    "opt.disableflags": mujoco.mjtDisableBit.mjDSBL_EULERDAMP,
                    "opt.iterations": 1,
                    "opt.ls_iterations": 4,
                }
            )

        if backend == "positional":
            # TODO: does the same actuator strength work as in spring
            sys = sys.replace(actuator=sys.actuator.replace(gear=200 * jnp.ones_like(sys.actuator.gear)))

        kwargs["n_frames"] = kwargs.get("n_frames", n_frames)

        super().__init__(sys=sys, backend=backend, **kwargs)

        self._ctrl_cost_weight = ctrl_cost_weight
        self._use_contact_forces = use_contact_forces
        self._contact_cost_weight = contact_cost_weight
        self._healthy_reward = healthy_reward
        self._terminate_when_unhealthy = terminate_when_unhealthy
        self._healthy_z_range = healthy_z_range
        self._contact_force_range = contact_force_range
        self._reset_noise_scale = reset_noise_scale
        self._exclude_current_positions_from_observation = exclude_current_positions_from_observation
        self.dense_reward = dense_reward
        # Each agent owns a block that mirrors the single-agent observation:
        #   [qpos_i (15, or 13 if exclude_current_positions), qvel_i (14),
        #    skill one-hot (4 if applicable), target xy (2)]
        # The full observation is just these blocks concatenated.
        per_agent_qpos_dim = self._qpos_per_agent - (
            2 if exclude_current_positions_from_observation else 0
        )
        per_agent_obs_dim = per_agent_qpos_dim + self._qvel_per_agent + 2
        if self.skill_mazes or self.skill_mazes_eval:
            per_agent_obs_dim += 4
        self._per_agent_obs_dim = per_agent_obs_dim
        self.state_dim = self._n_agents * per_agent_obs_dim
        # Ant A's xy is the first two entries of its own block (which is the first block).
        self.goal_indices = jnp.array([0, 1])
        self.goal_reach_thresh = 0.5

        if self._use_contact_forces:
            raise NotImplementedError("use_contact_forces not implemented.")

    def reset(self, rng: jax.Array) -> State:
        """Resets the environment to an initial state with a randomly sampled skill."""
        skill_idx = jnp.int32(0)
        if self.skill_mazes:
            rng, rng1, rng2, rng3, rng4 = jax.random.split(rng, 5)
            skill_idx = jax.random.randint(rng4, (1,), 0, 4)[0]
        else:
            rng, rng1, rng2, rng3 = jax.random.split(rng, 4)
        return self._reset_impl(rng, rng1, rng2, rng3, skill_idx)

    def reset_with_skill(self, rng: jax.Array, skill_idx: jax.Array) -> State:
        """Reset with a specific skill index (JIT-compatible).

        Unlike ``reset``, the skill is not sampled randomly but taken from
        ``skill_idx`` which can be a traced JAX value.  The RNG split pattern
        is kept identical to ``reset`` so downstream randomness is unchanged.
        """
        skill_idx = jnp.int32(skill_idx)
        if self.skill_mazes or self.skill_mazes_eval:
            rng, rng1, rng2, rng3, rng4 = jax.random.split(rng, 5)
        else:
            rng, rng1, rng2, rng3 = jax.random.split(rng, 4)
        return self._reset_impl(rng, rng1, rng2, rng3, skill_idx)

    def _reset_impl(self, rng, rng1, rng2, rng3, skill_idx) -> State:
        """Shared reset body used by both ``reset`` and ``reset_with_skill``."""
        low, hi = -self._reset_noise_scale, self._reset_noise_scale
        q = self.sys.init_q + jax.random.uniform(rng, (self.sys.q_size(),), minval=low, maxval=hi)
        qd = hi * jax.random.normal(rng1, (self.sys.qd_size(),))

        # If the maze lists multiple R cells, each agent samples its own start uniformly.
        # With a single start cell and multiple agents, keep one sampled cell plus offsets.
        if self.possible_starts.shape[0] > 1:
            # Ensure that starts do not overlap: sample a permutation for agent starts
            start_indices = jax.random.permutation(rng2, self.possible_starts.shape[0])[:self._n_agents]
            for agent_i in range(self._n_agents):
                base = agent_i * self._qpos_per_agent
                start_i = self.possible_starts[start_indices[agent_i]]
                q = q.at[base : base + 2].set(start_i)
       
        else:
            start = self._random_start(rng2)
            for agent_i in range(self._n_agents):
                base = agent_i * self._qpos_per_agent
                q = q.at[base : base + 2].set(start)

        if self.skill_mazes or self.skill_mazes_eval:
            target = self._random_target_for_skill(rng3, skill_idx)
        else:
            target = self._random_target(rng3, skill_idx)
        q = q.at[-2:].set(target)

        qd = qd.at[-2:].set(0)

        pipeline_state = self.pipeline_init(q, qd)
        obs = self._get_obs(pipeline_state, skill_idx)

        if "single_goal" in self.maze_layout_name:
            self.hard_goal = jnp.array([12, 4])

        reward, done, zero = jnp.zeros(3)
        metrics = {
            "reward_forward": zero,
            "reward_survive": zero,
            "reward_ctrl": zero,
            "reward_contact": zero,
            "x_position": zero,
            "y_position": zero,
            "distance_from_origin": zero,
            "x_velocity": zero,
            "y_velocity": zero,
            "forward_reward": zero,
            "dist": zero,
            "success": zero,
            "success_easy": zero,
            "current_step": zero,
            "skill_idx": jnp.float32(skill_idx),
        }
        state = State(pipeline_state, obs, reward, done, metrics)
        return state

    def step(self, state: State, action: jax.Array) -> State:
        """Run one timestep of the environment's dynamics.

        The 1-D action vector has length ``n_agents * 8`` and is fed directly to the
        physics layer (the XML's actuator ordering already groups ant A's 8 motors
        first, then ant B's). All ants share a single goal and a single scalar reward.
        """
        pipeline_state0 = state.pipeline_state
        pipeline_state = self.pipeline_step(pipeline_state0, action)

        q0 = pipeline_state0.q
        q1 = pipeline_state.q
        target_pos = q1[-2:]

        # Per-agent xy positions before/after the step and z after the step.
        per_agent_xy0 = jnp.stack(
            [q0[i * self._qpos_per_agent : i * self._qpos_per_agent + 2] for i in range(self._n_agents)]
        )
        per_agent_xy1 = jnp.stack(
            [q1[i * self._qpos_per_agent : i * self._qpos_per_agent + 2] for i in range(self._n_agents)]
        )
        per_agent_z = jnp.stack(
            [q1[i * self._qpos_per_agent + 2] for i in range(self._n_agents)]
        )

        per_agent_vel = (per_agent_xy1 - per_agent_xy0) / self.dt  # (n_agents, 2)
        forward_reward = jnp.mean(per_agent_vel[:, 0])

        # Healthy = all ants within healthy_z_range. Treat 0/1 floats as booleans and AND them.
        min_z, max_z = self._healthy_z_range
        healthy_per_agent = jnp.where(per_agent_z < min_z, 0.0, 1.0)
        healthy_per_agent = jnp.where(per_agent_z > max_z, 0.0, healthy_per_agent)
        is_healthy = jnp.prod(healthy_per_agent)

        if self._terminate_when_unhealthy:
            healthy_reward = self._healthy_reward
        else:
            healthy_reward = self._healthy_reward * is_healthy
        ctrl_cost = self._ctrl_cost_weight * jnp.sum(jnp.square(action))
        contact_cost = 0.0

        skill_idx = state.metrics["skill_idx"].astype(jnp.int32)
        obs = self._get_obs(pipeline_state, skill_idx)

        # Per-agent distance to the shared goal. Cooperative reach-all: success only when every ant is at the goal.
        per_agent_dist0 = jnp.linalg.norm(per_agent_xy0 - target_pos, axis=-1)
        per_agent_dist1 = jnp.linalg.norm(per_agent_xy1 - target_pos, axis=-1)
        per_agent_success = jnp.array(per_agent_dist1 < self.goal_reach_thresh, dtype=float)
        per_agent_success_easy = jnp.array(per_agent_dist1 < 2.0, dtype=float)
        success = jnp.prod(per_agent_success)
        success_easy = jnp.prod(per_agent_success_easy)
        dist = jnp.mean(per_agent_dist1)
        vel_to_target = jnp.mean((per_agent_dist0 - per_agent_dist1) / self.dt)

        if self.dense_reward:
            reward = 10 * vel_to_target + healthy_reward - ctrl_cost - contact_cost
        else:
            reward = success

        done = 1.0 - is_healthy if self._terminate_when_unhealthy else 0.0

        mean_xy1 = jnp.mean(per_agent_xy1, axis=0)
        mean_vel = jnp.mean(per_agent_vel, axis=0)
        state.metrics.update(
            reward_forward=forward_reward,
            reward_survive=healthy_reward,
            reward_ctrl=-ctrl_cost,
            reward_contact=-contact_cost,
            x_position=mean_xy1[0],
            y_position=mean_xy1[1],
            distance_from_origin=math.safe_norm(mean_xy1),
            x_velocity=mean_vel[0],
            y_velocity=mean_vel[1],
            forward_reward=forward_reward,
            dist=dist,
            success=success,
            success_easy=success_easy,
            current_step=state.metrics["current_step"]+1,
        )
        return state.replace(pipeline_state=pipeline_state, obs=obs, reward=reward, done=done)

    def _get_obs(self, pipeline_state: base.State, skill_index) -> jax.Array:
        """Build the observation as per-agent blocks concatenated together.

        Each per-agent block has the same layout as the single-agent observation:
            [qpos_i (xy stripped if `exclude_current_positions`), qvel_i,
             skill one-hot (if applicable), target_xy]
        Goal and (optional) skill features are duplicated into every block so an
        agent-i policy can read its own block by simple slicing (see `split_per_agent`).
        """
        qpos_all = pipeline_state.q[:-2]
        qvel_all = pipeline_state.qd[:-2]
        target_pos = pipeline_state.x.pos[-1][:2]

        use_skill = self.skill_mazes or self.skill_mazes_eval
        skill_index_onehot = jax.nn.one_hot(jnp.int32(skill_index), 4) if use_skill else None

        per_agent_obs = []
        for i in range(self._n_agents):
            qpos_i = qpos_all[i * self._qpos_per_agent : (i + 1) * self._qpos_per_agent]
            qvel_i = qvel_all[i * self._qvel_per_agent : (i + 1) * self._qvel_per_agent]
            if self._exclude_current_positions_from_observation:
                qpos_i = qpos_i[2:]
            parts = [qpos_i, qvel_i]
            if use_skill:
                parts.append(skill_index_onehot)
            parts.append(target_pos)
            per_agent_obs.append(jnp.concatenate(parts))

        return jnp.concatenate(per_agent_obs)

    def split_per_agent(self, obs: jax.Array, action: jax.Array):
        """Split a joint observation/action pair into per-agent components.

        Both inputs are laid out as ``[agent_0 | agent_1 | ... | agent_{N-1}]`` along
        the trailing axis, so a simple reshape recovers per-agent rows. Works for a
        single sample (1-D inputs) as well as any leading batch dimensions.

        Args:
            obs:    array of shape ``(..., n_agents * per_agent_obs_dim)``.
            action: array of shape ``(..., n_agents * 8)``.

        Returns:
            ``(per_agent_obs, per_agent_action)`` with shapes
            ``(..., n_agents, per_agent_obs_dim)`` and ``(..., n_agents, 8)``.
            Agent ``i``'s slice is therefore ``per_agent_obs[..., i, :]`` (etc.).
        """
        per_agent_obs = obs.reshape(
            obs.shape[:-1] + (self._n_agents, self._per_agent_obs_dim)
        )
        per_agent_action = action.reshape(
            action.shape[:-1] + (self._n_agents, self._act_per_agent)
        )
        return per_agent_obs, per_agent_action

    def _random_target(self, rng: jax.Array, skill_idx) -> jax.Array:
        """Returns a random target location chosen from possibilities specified in the maze layout."""
        idx = jax.random.randint(rng, (1,), 0, len(self.possible_goals))
        return jnp.array(self.possible_goals[idx])[0]

    def _random_target_for_skill(self, rng: jax.Array, skill_idx) -> jax.Array:
        """Returns a random target from the goal set corresponding to skill_idx (JIT-safe)."""
        branches = []
        for goals in self.goals_skills:
            goals_array = jnp.array(goals)
            def branch_fn(rng, g=goals_array):
                idx = jax.random.randint(rng, (1,), 0, g.shape[0])
                return g[idx[0]]
            branches.append(branch_fn)
        skills_target = jax.lax.switch(jnp.int32(skill_idx), branches, rng)
        # jax.debug.print("skill_idx: {x}", x=skill_idx)
        # jax.debug.print("skills_target: {x}", x=skills_target)
        return skills_target
        

    def _random_start(self, rng: jax.Array) -> jax.Array:
        """Returns a random start location chosen from possibilities specified in the maze layout."""
        idx = jax.random.randint(rng, (1,), 0, len(self.possible_starts))
        return jnp.array(self.possible_starts[idx])[0]


In [21]:
env_name = "ant_big_maze_single_goal_many_starts"
backend = "spring"
skill_mazes = False
env = AntMaze(backend="spring", maze_layout_name=env_name[4:], skill_mazes=False, skill_mazes_eval=False, n_agents=3)

In [22]:
seed = 0
num_envs = 1
random.seed(seed)
np.random.seed(seed)
key = jax.random.PRNGKey(seed)
key, env_keys, act_key = jax.random.split(key, 3)
env_state = jax.jit(env.reset)(env_keys)
obs = env_state.obs
print(f"obs_shape: {obs.shape}")
action = jax.random.uniform(key, (env.action_size, ))
print(f"action_shape: {action.shape}")
# next_obs, reward, done, info = jax.jit(env.step)(env_state, action)
# print(f"next_obs_shape: {next_obs.shape}")
# print(f"reward_shape: {reward.shape}")
# print(f"done_shape: {done.shape}")
# print(f"info_shape: {info.shape}")


obs_shape: (93,)
action_shape: (24,)


In [23]:
env, rollout = collect_rollout(act_key, env)

In [24]:
exp_dir = "./multi_ant_rendering_videos"
exp_name = "multi_ant_maze_prototyping"
num_steps = 200
# url = html.render(env.sys.tree_replace({"opt.timestep": env.dt}), rollout, height=1024)
# maze_top is injected in make_maze (top-down, full maze). Per-ant: track_a / track_b.
frames = image.render_array(
    env.sys.tree_replace({"opt.timestep": env.dt}),
    rollout,
    height=480,
    width=640,
    camera="maze_top",
)
import imageio
imageio.mimsave(os.path.join(exp_dir, f"{exp_name}_{num_steps}.gif"), frames, duration=100)
# with open(os.path.join(exp_dir, f"{exp_name}_{num_steps}.html"), "w") as file:
#     file.write(url)
# print("experiment directory: ", exp_dir)